In [0]:
# Clear any stale library state before importing
dbutils.library.restartPython()

In [0]:
import sys

# music_pipeline_setup.py lives two levels up in 00_setup/pawel_project.
# Relative imports are unsupported in Databricks notebooks, and '00_setup' starts
# with digits (invalid Python identifier), so we add the directory to sys.path
# and import music_pipeline_setup as a regular module.
sys.path.insert(0, "../../00_setup/pawel_project")

In [0]:
from music_pipeline_setup import bronze_music_metadata_table


metadata_table = spark.read.table(bronze_music_metadata_table)

In [0]:
# Inspect the raw metadata rows.
display(metadata_table)

In [0]:
# Enrich the metadata table with a ``yt_key`` column parsed from the YouTube URL.
from pyspark.sql.functions import col, regexp_extract


metadata_table = metadata_table.withColumn(
    "yt_key", 
    regexp_extract(col("url"), r"v=([a-zA-Z0-9_-]{11})", 1)
)

In [0]:
# Verify yt_key extraction on a sample row.
display(metadata_table.show(1))

In [0]:
# Collect distinct non-null yt_key values into a Python list for batched API calls.
yt_key_rows = metadata_table.select("yt_key").distinct().collect()

video_ids_list = [row.yt_key for row in yt_key_rows if row.yt_key]

In [0]:
import requests
from music_pipeline_setup import yt_api_key
from datetime import datetime

url: str = "https://www.googleapis.com/youtube/v3/videos"

# The youtube API can take up to 50 videos per batch (more videos will result in 400 error code by API)
BATCH_SIZE = 50

# Precompute yt_key -> album mapping for fast lookup
yt_key_album_map = {
    row.yt_key: row.album
    for row in metadata_table.select("yt_key", "album")
    .filter(col("yt_key").isNotNull())
    .toLocalIterator()
}

# Split the videos into 50-sized batches
id_batches = [video_ids_list[i:i + BATCH_SIZE] for i in range(0, len(video_ids_list)+1, BATCH_SIZE)]

all_video_responses = []

for idx, batch in enumerate(id_batches):
    batch_ids_str = ",".join(batch)
    
    params = {
        "part": "snippet, statistics",
        "id": batch_ids_str,
        "key": yt_api_key
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        items = response.json().get("items", [])

        for item in items:
            item_snippet = item.get("snippet", {})
            item_statistics = item.get("statistics", {})

            yt_key = item.get("id")
            published_at:datetime = item_snippet.get("publishedAt")
            title:str = item_snippet.get("title")
            channel_title:str = item_snippet.get("channelTitle")

            viewCount:str = item_statistics.get("viewCount")
            likeCount:str = item_statistics.get("likeCount")
            dislikeCount:str = item_statistics.get("dislikeCount")
            commentCount:str = item_statistics.get("commentCount")

            album = yt_key_album_map.get(yt_key)

            video_response = {
                "yt_key": yt_key,
                "album": album,
                "published_at": published_at,
                "title": title,
                "channel_title": channel_title,
                "viewCount": viewCount,
                "likeCount": likeCount,
                "commentCount": commentCount
            }
            all_video_responses.append(video_response)

    else:
        print("Error ")

In [0]:
# Sanity-check the first returned record before persisting.
print(all_video_responses[0])

In [0]:
import json
from music_pipeline_setup import json_landing_path
from datetime import datetime 

dbutils.fs.mkdirs(json_landing_path)

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S") # Ingestion datetime
json_file_path = f"{json_landing_path}/yt_stats_{timestamp_str}.json"

with open(json_file_path, "w", encoding="utf-8") as f:
    json.dump(all_video_responses, f, ensure_ascii=False, indent=4)

# To Do
* Refactor this notebook into importable Python scripts for the bronze ingestion pipeline.